# 🩺 Multilingual Health QA — Checkpointed Retrieval Pipeline

**Approach:** Semantic embedding + TF-IDF hybrid retrieval (no GPU fine-tuning required for most experiments). For any test/val question, encode it, find the most similar question in the train+val corpus, return *that* question's answer. This is fast (~minutes, not hours), avoids OOM entirely, and is a legitimate, well-documented IR-based QA strategy.


## Experiment plan (10 total — mix of retrieval + 2 mT5 fine-tunes)

| # | ID | Name | Time | GPU needed? |
|---|----|------|------|---|
| 1 | E01 | TF-IDF only baseline | ~10 min | No |
| 2 | E02 | Semantic-only (sentence-transformers) | ~15 min | Helps, not required |
| 3 | E03 | Hybrid (TF-IDF + semantic, fixed weights) | ~15 min | Helps |
| 4 | E04 | Per-subset weight tuning (grid search) | ~30 min | Helps |
| 5 | E05 | Exact-match lookup short-circuit | ~10 min | No |
| 6 | E06 | Dedup near-identical questions before indexing | ~10 min | No |
| 7 | E07 | Char-level TF-IDF for low-resource subsets | ~15 min | No |
| 8 | E08 | Cross-subset fallback (e.g. Amh→Eng) | ~15 min | No |
| 9 | E09 | mT5-small fine-tune, 2 epochs, 25% data | ~40 min | **Yes** |
| 10 | E10 | Best retrieval config + mT5 ensemble/compare | ~20 min | Maybe |

Each is independent — you can do them in any order, 1–2 per session.

## Section 1 — Install & Imports

## Environment Setup (Colab + Kaggle compatible)

This notebook runs unmodified on both Google Colab and Kaggle. On Colab, it mounts your Google Drive and looks for the competition CSVs there. On Kaggle, it uses the attached competition dataset as before.

**Colab users:** upload `Train.csv`, `Val.csv`, `Test.csv`, `SampleSubmission.csv` to a folder in your Google Drive (e.g. `MyDrive/health-qa-data/`) before running. When prompted below, authorize Drive access.

In [ ]:
import os

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    print("Running on Google Colab — mounting Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    WORKING_DIR = '/content/outputs'
    os.makedirs(WORKING_DIR, exist_ok=True)
else:
    print("Running on Kaggle (or other non-Colab environment).")
    WORKING_DIR = '/kaggle/working'

print(f"Working directory for outputs: {WORKING_DIR}")

In [1]:
%%capture
!pip install -q sentence-transformers rouge-score evaluate scikit-learn

In [2]:
import os
import re
import json
import time
import pickle
import shutil
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU — retrieval experiments still work fine, just slower to encode.")

Device: cuda
GPU: Tesla T4


## Section 2 — Auto-locate Data & Config

In [3]:
def find_data_dir(filename: str = 'Train.csv', search_roots=None) -> Path:
    """Searches Kaggle's /kaggle/input AND Colab's mounted Drive automatically —
    works unmodified on both platforms. On Colab, searches all of /content/drive,
    so your data can live in any subfolder of MyDrive without editing this notebook."""
    if search_roots is None:
        if IN_COLAB:
            search_roots = ('/content/drive/MyDrive', '/content/drive/Shareddrives', '.')
        else:
            search_roots = ('/kaggle/input', '.')

    for root in search_roots:
        if not os.path.exists(root):
            continue
        for dirpath, _, files in os.walk(root):
            if filename in files:
                return Path(dirpath)
    raise FileNotFoundError(
        f"Could not find '{filename}' under {search_roots}. "
        f"{'Upload your competition CSVs to a folder in your Google Drive (e.g. MyDrive/health-qa-data/).' if IN_COLAB else "Attach your competition dataset via 'Add Input' in the Kaggle sidebar."}"
    )

DATA_DIR = find_data_dir('Train.csv')
print(f"✅ Data directory: {DATA_DIR}")

TRAIN_PATH  = DATA_DIR / 'Train.csv'
VAL_PATH    = DATA_DIR / 'Val.csv'
TEST_PATH   = DATA_DIR / 'Test.csv'
SAMPLE_PATH = DATA_DIR / 'SampleSubmission.csv'

# Checkpoint directory — this is what you zip up and re-upload between sessions.
# Uses WORKING_DIR (set in the environment-setup cell above), so this resolves
# correctly to /content/outputs on Colab or /kaggle/working on Kaggle.
CKPT_DIR = Path(WORKING_DIR) / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# If you've re-attached a previous checkpoint dataset/folder, point this at it.
# Section 3b (checkpoint loader) auto-searches both platforms' typical locations.
PREV_CKPT_SEARCH_ROOTS = ('/content/drive/MyDrive', '.') if IN_COLAB else ('/kaggle/input',)

QUESTION_COL = 'input'
ANSWER_COL   = 'output'
LANG_COL     = 'subset'
ID_COL       = 'ID'

TEST_QUESTION_COL = 'input'
TEST_LANG_COL     = 'subset'
TEST_ID_COL       = 'ID'

SEMANTIC_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
SEMANTIC_BATCH_SIZE = 64

print(f"Question col: {QUESTION_COL} | Answer col: {ANSWER_COL} | Lang col: {LANG_COL}")

✅ Data directory: /kaggle/input/datasets/renepntabana/multilingual-dataset
Question col: input | Answer col: output | Lang col: subset


## Section 3 — Load Data

In [4]:
def clean_text(x) -> str:
    if pd.isna(x):
        return ''
    return re.sub(r'\s+', ' ', str(x).strip())


train = pd.read_csv(TRAIN_PATH)
val   = pd.read_csv(VAL_PATH)
test  = pd.read_csv(TEST_PATH)
sample_df = pd.read_csv(SAMPLE_PATH)

for df in (train, val, test):
    df[QUESTION_COL] = df[QUESTION_COL].map(clean_text)
for df in (train, val):
    df[ANSWER_COL] = df[ANSWER_COL].map(clean_text)

train = train[(train[QUESTION_COL] != '') & (train[ANSWER_COL] != '')].reset_index(drop=True)
val   = val[(val[QUESTION_COL] != '') & (val[ANSWER_COL] != '')].reset_index(drop=True)
test  = test[test[QUESTION_COL] != ''].reset_index(drop=True)

print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
print(f"\nTrain subsets:\n{train[LANG_COL].value_counts()}")

Train: 29,814 | Val: 6,686 | Test: 2,618

Train subsets:
subset
Eng_Uga    7623
Aka_Gha    4455
Eng_Gha    4443
Eng_Eth    3915
Lug_Uga    3383
Eng_Ken    2080
Swa_Ken    2070
Amh_Eth    1845
Name: count, dtype: int64


## Section 3b — Load Checkpoint (run this if resuming a previous session)

If you've attached a previously-saved checkpoint dataset via **Add Input**, this cell restores it automatically — saving you from re-encoding embeddings (the slowest step). If no checkpoint is found, it just continues with an empty tracker, which is correct for your very first session.

In [5]:
def find_checkpoint_dir(filename='experiment_tracker.csv', search_roots=PREV_CKPT_SEARCH_ROOTS):
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for dirpath, _, files in os.walk(root):
            if filename in files and 'checkpoints' in dirpath.lower() or filename in files:
                # avoid accidentally matching the competition data dir
                if dirpath == str(DATA_DIR):
                    continue
                return Path(dirpath)
    return None


PREV_CKPT_DIR = find_checkpoint_dir('experiment_tracker.csv')

if PREV_CKPT_DIR:
    print(f"✅ Found previous checkpoint at: {PREV_CKPT_DIR}")
    for f in sorted(os.listdir(PREV_CKPT_DIR)):
        print(f"   {f}")
    # Copy everything into the working checkpoint dir so save logic can append to it
    for f in os.listdir(PREV_CKPT_DIR):
        src = PREV_CKPT_DIR / f
        dst = CKPT_DIR / f
        if src.is_file():
            shutil.copy(src, dst)
    print(f"\nCopied into working checkpoint dir: {CKPT_DIR}")
else:
    print("No previous checkpoint found — this looks like your first session. That's fine.")

No previous checkpoint found — this looks like your first session. That's fine.


## Section 4 — Experiment Tracker

This small class appends one row per experiment to a CSV. It auto-loads existing rows from the checkpoint (if any), so re-running this cell across sessions never loses history.

In [6]:
TRACKER_PATH = CKPT_DIR / 'experiment_tracker.csv'

TRACKER_COLUMNS = [
    'timestamp', 'experiment_id', 'name', 'category', 'change', 'rationale',
    'rouge1', 'rougel', 'lb_score', 'runtime_min', 'notes',
]


class ExperimentTracker:
    def __init__(self, path: Path):
        self.path = path
        if path.exists():
            self.df = pd.read_csv(path)
            print(f"Loaded existing tracker with {len(self.df)} experiment(s).")
        else:
            self.df = pd.DataFrame(columns=TRACKER_COLUMNS)
            print("Starting a fresh experiment tracker.")

    def log(self, experiment_id, name, category, change, rationale,
            rouge1=None, rougel=None, lb_score=None, runtime_min=None, notes=''):
        row = {
            'timestamp': datetime.now().isoformat(timespec='seconds'),
            'experiment_id': experiment_id,
            'name': name,
            'category': category,
            'change': change,
            'rationale': rationale,
            'rouge1': rouge1,
            'rougel': rougel,
            'lb_score': lb_score,
            'runtime_min': runtime_min,
            'notes': notes,
        }
        # Overwrite if same experiment_id is re-run, else append
        self.df = self.df[self.df['experiment_id'] != experiment_id]
        self.df = pd.concat([self.df, pd.DataFrame([row])], ignore_index=True)
        self.save()
        print(f"Logged {experiment_id}: {name} | ROUGE-1={rouge1} ROUGE-L={rougel}")

    def save(self):
        self.df.to_csv(self.path, index=False)

    def show(self):
        if len(self.df) == 0:
            print("No experiments logged yet.")
            return
        display_cols = ['experiment_id', 'name', 'rouge1', 'rougel', 'lb_score', 'runtime_min']
        print(self.df[display_cols].to_string(index=False))


tracker = ExperimentTracker(TRACKER_PATH)
tracker.show()

Starting a fresh experiment tracker.
No experiments logged yet.


### Experiment Config Registry (init)

This dict accumulates `{experiment_id: {'language_strategy':..., 'index_kwargs':...}}` as each experiment cell runs below — letting `submit_experiment(experiment_id)` near the end rebuild any past experiment's exact config without you retyping it.

In [7]:
if 'EXPERIMENT_CONFIGS' not in dir():
    EXPERIMENT_CONFIGS = {}

print(f"Configs currently available: {list(EXPERIMENT_CONFIGS.keys())}")

Configs currently available: []


## Section 5 — ROUGE Scoring & Language Strategy

In [8]:
class WhitespaceTokenizer:
    """Whitespace tokeniser for ROUGE — language-agnostic, matches the official
    Zindi starter notebook's scoring approach so offline numbers are comparable."""
    def tokenize(self, text):
        return [] if text is None else str(text).strip().split()


_scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rougeL'], tokenizer=WhitespaceTokenizer(), use_stemmer=False
)


def compute_rouge(predictions, references):
    r1, rl = [], []
    for p, r in zip(predictions, references):
        s = _scorer.score(str(r), str(p))
        r1.append(s['rouge1'].fmeasure)
        rl.append(s['rougeL'].fmeasure)
    return {'rouge1_f1': float(np.mean(r1)), 'rougeL_f1': float(np.mean(rl))}


def print_run_summary(label, predictions, references, languages=None, min_answer_len=None):
    metrics = compute_rouge(predictions, references)
    print(f"\n=== {label} ===")
    print(f"ROUGE-1 F1: {metrics['rouge1_f1']:.4f}")
    print(f"ROUGE-L F1: {metrics['rougeL_f1']:.4f}")
    if languages is not None:
        tmp = pd.DataFrame({'pred': predictions, 'ref': references, 'lang': languages})
        per_lang = []
        for lang, group in tmp.groupby('lang'):
            m = compute_rouge(group['pred'].tolist(), group['ref'].tolist())
            per_lang.append({'subset': lang, 'n': len(group), **m})
        per_lang_df = pd.DataFrame(per_lang).sort_values('n', ascending=False)
        print(per_lang_df.to_string(index=False))
    if min_answer_len is not None:
        short = sum(1 for p in predictions if len(str(p)) < min_answer_len)
        print(f"Predictions shorter than {min_answer_len} chars: {short}/{len(predictions)}")
    return metrics


# Default per-subset strategy: (method, tfidf_weight, semantic_weight)
# 'hybrid' blends both; 'semantic' or 'tfidf' use tfidf_w=0 / semantic_w=0 respectively.
LANGUAGE_STRATEGY = {
    subset: ('hybrid', 0.35, 0.65) for subset in train['subset'].unique()
}

print("ROUGE scorer + LANGUAGE_STRATEGY ready.")
print(f"Subsets configured: {list(LANGUAGE_STRATEGY.keys())}")

ROUGE scorer + LANGUAGE_STRATEGY ready.
Subsets configured: ['Aka_Gha', 'Amh_Eth', 'Eng_Eth', 'Eng_Gha', 'Eng_Ken', 'Eng_Uga', 'Lug_Uga', 'Swa_Ken']


## Section 6 — Semantic Retrieval Index

This is the core class — adapted from a peer's higher-scoring approach. It builds a per-subset hybrid (TF-IDF + sentence-embedding) nearest-neighbour index over the training questions, then answers new questions by copying the answer from the most similar training question.

In [9]:
# (SentenceTransformer already imported in Section 6 above)

In [10]:
# Module-level encoder cache, keyed by model name. Every SemanticRoutingIndex
# instance shares the SAME loaded SentenceTransformer weights instead of each
# loading its own copy onto the GPU — this is what was silently eating ~3.7GB
# per experiment (8 experiments x ~470MB encoder = ~3.7GB never released).
_SHARED_ENCODERS = {}


def _get_shared_encoder(model_name):
    if model_name not in _SHARED_ENCODERS:
        enc_device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f'  Loading encoder (shared, cached): {model_name} ({enc_device})')
        _SHARED_ENCODERS[model_name] = SentenceTransformer(model_name, device=enc_device)
    return _SHARED_ENCODERS[model_name]


def release_shared_encoders():
    """Call this before E09 (or any GPU-heavy step) to free the cached encoder(s)
    from GPU memory once retrieval experiments are done with them."""
    global _SHARED_ENCODERS
    for name, enc in _SHARED_ENCODERS.items():
        del enc
    _SHARED_ENCODERS = {}
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Released all cached sentence-transformer encoders from GPU memory.")


def _norm_question_key(q):
    return clean_text(q).lower()


def _minmax_scores(scores):
    scores = np.asarray(scores, dtype=float)
    lo, hi = scores.min(), scores.max()
    if hi - lo < 1e-9:
        return np.ones_like(scores)
    return (scores - lo) / (hi - lo)


class SemanticRoutingIndex:
    """Per-subset semantic + TF-IDF top-1 retrieval with ablation hooks.

    Adapted from a peer's higher-scoring solution. Fits on a corpus of
    (question, answer, subset) rows; at inference time, encodes a new
    question, finds the most similar question within the same subset
    (optionally falling back to a related subset), and returns that
    question's answer verbatim.
    """

    def __init__(
        self,
        model_name=SEMANTIC_MODEL_NAME,
        batch_size=SEMANTIC_BATCH_SIZE,
        language_strategy=None,
        question_col=QUESTION_COL,
        answer_col=ANSWER_COL,
        id_col='ID',
        group_col=LANG_COL,
        tfidf_analyzer='word',
        tfidf_ngram_range=None,
        tfidf_max_features=200_000,
        normalize_hybrid_scores=False,
        exact_match_lookup=False,
        dedup_questions=False,
        similarity_threshold=None,
        cross_subset_fallback=None,
        char_tfidf_subsets=None,
        query_encode_prefix='',
        passage_encode_prefix='',
    ):
        self.model_name = model_name
        self.batch_size = batch_size
        self.language_strategy = language_strategy or LANGUAGE_STRATEGY
        self.question_col = question_col
        self.answer_col = answer_col
        self.id_col = id_col
        self.group_col = group_col
        self.tfidf_analyzer = tfidf_analyzer
        self.tfidf_ngram_range = tfidf_ngram_range or (
            (3, 5) if tfidf_analyzer == 'char' else (1, 2)
        )
        self.tfidf_max_features = tfidf_max_features
        self.normalize_hybrid_scores = normalize_hybrid_scores
        self.exact_match_lookup = exact_match_lookup
        self.dedup_questions = dedup_questions
        self.similarity_threshold = similarity_threshold
        self.cross_subset_fallback = cross_subset_fallback or {}
        self.char_tfidf_subsets = set(char_tfidf_subsets or ())
        self.query_encode_prefix = query_encode_prefix
        self.passage_encode_prefix = passage_encode_prefix
        self.encoder = None
        self.questions = []
        self.answers = []
        self.ids = []
        self.embeddings = None
        self.subset_indices = {}
        self.subset_tfidf = {}
        self.question_to_answer = {}

    def _get_encoder(self):
        # FIX: previously this loaded a brand-new SentenceTransformer onto the GPU
        # for every single SemanticRoutingIndex instance — E01 through E08 each
        # created their own index, so by E09 there were 8 separate uncollected
        # copies of the same ~470MB encoder sitting in GPU memory, leaving almost
        # nothing free for mt5 training. Fix: cache by model_name in a module-level
        # dict so every experiment reuses the SAME loaded weights instead of
        # loading a fresh copy each time.
        if self.encoder is None:
            self.encoder = _get_shared_encoder(self.model_name)
        return self.encoder

    def _encode_texts(self, texts, prefix=''):
        encoder = self._get_encoder()
        payload = [f"{prefix}{t}" if prefix else t for t in texts]
        parts = []
        for start in range(0, len(payload), self.batch_size):
            end = min(start + self.batch_size, len(payload))
            batch = payload[start:end]
            parts.append(
                encoder.encode(batch, show_progress_bar=False, normalize_embeddings=True)
            )
            if end % 5000 == 0 or end == len(payload):
                print(f'    Encoded {end:,} / {len(payload):,}')
        return np.vstack(parts)

    def _make_tfidf_vectorizer(self):
        if self.tfidf_analyzer == 'char':
            return TfidfVectorizer(
                analyzer='char',
                ngram_range=self.tfidf_ngram_range,
                max_features=self.tfidf_max_features,
            )
        return TfidfVectorizer(
            ngram_range=self.tfidf_ngram_range,
            max_features=self.tfidf_max_features,
        )

    def fit(self, df):
        work = df.copy()
        if self.dedup_questions:
            work['_q_norm'] = work[self.question_col].map(clean_text)
            work = (
                work
                .drop_duplicates(subset=[self.group_col, '_q_norm'], keep='first')
                .drop(columns=['_q_norm'])
                .reset_index(drop=True)
            )
        self.questions = work[self.question_col].fillna('').astype(str).tolist()
        self.answers = work[self.answer_col].fillna('').astype(str).tolist()
        self.ids = (
            work[self.id_col].fillna('').astype(str).tolist()
            if self.id_col in work.columns else [''] * len(self.questions)
        )
        print(f'  Encoding {len(self.questions):,} questions...')
        self.embeddings = self._encode_texts(self.questions, prefix=self.passage_encode_prefix)
        if self.exact_match_lookup:
            self.question_to_answer = {}
            for q, a in zip(self.questions, self.answers):
                key = _norm_question_key(q)
                if key and key not in self.question_to_answer:
                    self.question_to_answer[key] = a
        self.subset_indices = {}
        self.subset_tfidf = {}
        for subset_code in work[self.group_col].unique():
            mask = work[self.group_col] == subset_code
            indices = np.where(mask.values)[0]
            self.subset_indices[subset_code] = indices
            subset_questions = [self.questions[i] for i in indices]
            if subset_code in self.char_tfidf_subsets:
                vectorizer = TfidfVectorizer(
                    analyzer='char',
                    ngram_range=(3, 5),
                    max_features=self.tfidf_max_features,
                )
            else:
                vectorizer = self._make_tfidf_vectorizer()
            matrix = vectorizer.fit_transform(subset_questions)
            self.subset_tfidf[subset_code] = {
                'vectorizer': vectorizer,
                'matrix': matrix,
            }
        print(
            f'  Built semantic index: {len(self.questions):,} rows, '
            f'{len(self.subset_indices)} subsets'
        )
        return self

    def _search_subsets(self, subset):
        if subset not in self.subset_indices:
            subset = next(iter(self.subset_indices))
        subsets = [subset]
        fallback = self.cross_subset_fallback.get(subset)
        if fallback and fallback in self.subset_indices and fallback not in subsets:
            subsets.append(fallback)
        return subsets

    def _collect_candidates(self, subsets, exclude_id=None):
        keep_global = []
        keep_local_by_subset = {}
        seen = set()
        for sub in subsets:
            all_indices = self.subset_indices[sub]
            keep_local = []
            for local_row, global_idx in enumerate(all_indices):
                if exclude_id is not None and self.ids[global_idx] == str(exclude_id):
                    continue
                gi = int(global_idx)
                if gi in seen:
                    continue
                seen.add(gi)
                keep_local.append(local_row)
                keep_global.append(gi)
            keep_local_by_subset[sub] = keep_local
        if not keep_global:
            sub = subsets[0]
            keep_global = [int(i) for i in self.subset_indices[sub]]
            keep_local_by_subset[sub] = list(range(len(keep_global)))
        return np.array(keep_global, dtype=int), keep_local_by_subset, subsets

    def retrieve_one(self, question, question_embedding, subset, exclude_id=None):
        if self.exact_match_lookup:
            hit = self.question_to_answer.get(_norm_question_key(question))
            if hit is not None:
                return hit
        subsets = self._search_subsets(subset)
        primary_subset = subsets[0]
        method, tfidf_w, semantic_w = self.language_strategy.get(
            primary_subset, ('hybrid', 0.35, 0.65)
        )
        candidate_indices, keep_local_by_subset, subsets = self._collect_candidates(
            subsets, exclude_id=exclude_id
        )
        candidate_embeddings = self.embeddings[candidate_indices]
        semantic_scores = cosine_similarity(
            question_embedding.reshape(1, -1), candidate_embeddings
        ).flatten()
        if method == 'semantic' or tfidf_w == 0.0:
            best_idx = int(candidate_indices[int(np.argmax(semantic_scores))])
            return self.answers[best_idx]
        subset_info = self.subset_tfidf[primary_subset]
        query_tfidf = subset_info['vectorizer'].transform([question])
        tfidf_scores = np.zeros(len(candidate_indices), dtype=float)
        offset = 0
        for sub in subsets:
            local_rows = keep_local_by_subset.get(sub, [])
            if not local_rows:
                continue
            n = len(local_rows)
            if sub == primary_subset:
                block = cosine_similarity(
                    query_tfidf, subset_info['matrix'][local_rows]
                ).flatten()
            else:
                fb_info = self.subset_tfidf[sub]
                block = cosine_similarity(
                    fb_info['vectorizer'].transform([question]),
                    fb_info['matrix'][local_rows],
                ).flatten()
            tfidf_scores[offset:offset + n] = block
            offset += n
        if self.normalize_hybrid_scores:
            semantic_scores = _minmax_scores(semantic_scores)
            tfidf_scores = _minmax_scores(tfidf_scores)
        if self.similarity_threshold is not None:
            if float(np.max(semantic_scores)) < self.similarity_threshold:
                tfidf_w, semantic_w = 1.0, 0.0
        hybrid_scores = tfidf_w * tfidf_scores + semantic_w * semantic_scores
        best_local = int(np.argmax(hybrid_scores))
        best_idx = int(candidate_indices[best_local])
        return self.answers[best_idx]

    def predict_dataframe(
        self,
        df,
        question_col,
        group_col,
        id_col=None,
        question_embeddings=None,
        desc='Semantic routing',
        log_every=500,
        references=None,
    ):
        questions = df[question_col].fillna('').astype(str).tolist()
        subsets = df[group_col].tolist()
        ids = (
            df[id_col].fillna('').astype(str).tolist()
            if id_col and id_col in df.columns else [None] * len(df)
        )
        if question_embeddings is None:
            print(f'  Encoding {len(questions):,} query embeddings...')
            question_embeddings = self._encode_texts(
                questions, prefix=self.query_encode_prefix
            )
        predictions = []
        pairs = list(zip(questions, subsets, ids, question_embeddings))
        row_iter = tqdm(pairs, total=len(pairs), desc=desc) if len(pairs) > 100 else pairs
        for i, (question, subset, row_id, emb) in enumerate(row_iter):
            predictions.append(
                self.retrieve_one(question, emb, subset, exclude_id=row_id)
            )
            if (
                references is not None
                and compute_rouge
                and log_every
                and (i + 1) % log_every == 0
            ):
                partial = compute_rouge(predictions, references[: len(predictions)])
                msg = (
                    f'  [{i + 1:,}/{len(df):,}] R1={partial["rouge1_f1"]:.4f} '
                    f'RL={partial["rougeL_f1"]:.4f}'
                )
                if len(pairs) > 100:
                    tqdm.write(msg)
                else:
                    print(msg)
        return predictions


print('SemanticRoutingIndex defined.')

SemanticRoutingIndex defined.


## Section 7 — Experiment Runner Helpers

`eval_semantic_experiment` fits an index on Train and scores it on the **official Val.csv** (never trained on), then logs the result to your experiment tracker automatically. `load_or_compute_val_embeddings()` caches the (slow) embedding step so repeat experiments in the same session are fast.

In [11]:
from sentence_transformers import SentenceTransformer

EMBEDDINGS_CACHE_PATH = CKPT_DIR / 'embeddings_cache.pkl'


def encode_questions_for_model(model_name, texts, batch_size=SEMANTIC_BATCH_SIZE, query_prefix=''):
    # FIX: reuse the same shared/cached encoder as SemanticRoutingIndex instead of
    # loading a brand-new SentenceTransformer on every call — this function gets
    # called repeatedly (once per experiment for val embeddings, once for test
    # embeddings during submission export), and each call was leaving an
    # uncollected model copy on the GPU.
    encoder = _get_shared_encoder(model_name)
    payload = [f"{query_prefix}{t}" if query_prefix else t for t in texts]
    parts = []
    for start in range(0, len(payload), batch_size):
        end = min(start + batch_size, len(payload))
        parts.append(
            encoder.encode(payload[start:end], show_progress_bar=False, normalize_embeddings=True)
        )
    return np.vstack(parts)


def load_or_compute_val_embeddings():
    """Val-question embeddings under the default semantic model are reused across
    almost every retrieval experiment — compute once per session (or load from a
    restored checkpoint) instead of re-encoding for every single experiment."""
    if EMBEDDINGS_CACHE_PATH.exists():
        print(f"Loading cached val embeddings from {EMBEDDINGS_CACHE_PATH} ...")
        with open(EMBEDDINGS_CACHE_PATH, 'rb') as f:
            cache = pickle.load(f)
        if cache.get('n_rows') == len(val) and cache.get('model_name') == SEMANTIC_MODEL_NAME:
            print(f"✅ Cache matches current Val.csv ({len(val)} rows) — reusing.")
            return cache['embeddings']
        print("Cache found but doesn't match current val set/model — recomputing.")

    val_qs = val[QUESTION_COL].fillna('').astype(str).tolist()
    print(f"Encoding {len(val_qs):,} val questions with {SEMANTIC_MODEL_NAME} ...")
    t0 = time.time()
    embeddings = encode_questions_for_model(SEMANTIC_MODEL_NAME, val_qs)
    print(f"Done in {(time.time() - t0) / 60:.1f} min.")

    with open(EMBEDDINGS_CACHE_PATH, 'wb') as f:
        pickle.dump({'embeddings': embeddings, 'n_rows': len(val), 'model_name': SEMANTIC_MODEL_NAME}, f)
    print(f"Cached to {EMBEDDINGS_CACHE_PATH} for future sessions.")
    return embeddings


def eval_semantic_experiment(
    experiment_id, name, change, rationale,
    index_kwargs=None, language_strategy=None, train_df=None,
    val_embeddings=None, category='semantic', notes='', desc=None,
    log_result=True, index=None,
):
    """Fit on train, evaluate on Val (honest protocol — never trained on Val),
    and log the result to the experiment tracker."""
    index_kwargs = dict(index_kwargs or {})
    if language_strategy is not None:
        index_kwargs['language_strategy'] = language_strategy
    train_df = train_df if train_df is not None else train

    t0 = time.time()
    if index is None:
        index = SemanticRoutingIndex(**index_kwargs).fit(train_df.copy())
    fit_min = (time.time() - t0) / 60

    model_name = index_kwargs.get('model_name', SEMANTIC_MODEL_NAME)
    query_prefix = index_kwargs.get('query_encode_prefix', '')
    emb = val_embeddings
    if emb is None or model_name != SEMANTIC_MODEL_NAME or query_prefix:
        val_qs = val[QUESTION_COL].fillna('').astype(str).tolist()
        emb = encode_questions_for_model(model_name, val_qs, query_prefix=query_prefix)

    t1 = time.time()
    preds = index.predict_dataframe(
        val, question_col=QUESTION_COL, group_col=LANG_COL, id_col='ID',
        question_embeddings=emb, desc=desc or f'Val {experiment_id}',
        log_every=0, references=val[ANSWER_COL].tolist(),
    )
    eval_min = (time.time() - t1) / 60

    metrics = print_run_summary(
        f'{experiment_id}: {name}', preds, val[ANSWER_COL].tolist(),
        languages=val[LANG_COL].tolist(), min_answer_len=60,
    )

    if log_result:
        tracker.log(
            experiment_id=experiment_id, name=name, category=category,
            change=change, rationale=rationale,
            rouge1=round(metrics['rouge1_f1'], 4), rougel=round(metrics['rougeL_f1'], 4),
            runtime_min=round(fit_min + eval_min, 1), notes=notes,
        )

    return metrics, index


def tune_hybrid_weights(index, val_embeddings, subsets_to_tune, weight_candidates=(0.0, 0.25, 0.5, 0.75, 1.0)):
    """Per-subset grid search over TF-IDF weight. Index must already be fitted."""
    tuned = dict(index.language_strategy)
    for subset in subsets_to_tune:
        mask = (val[LANG_COL] == subset).values
        if not mask.any():
            continue
        subset_val = val.loc[mask].reset_index(drop=True)
        subset_emb = val_embeddings[mask]
        best_w, best_r1 = tuned[subset][1], -1.0
        for tfidf_w in weight_candidates:
            semantic_w = 1.0 - tfidf_w
            method = 'semantic' if tfidf_w == 0.0 else ('tfidf' if semantic_w == 0.0 else 'hybrid')
            trial = dict(index.language_strategy)
            trial[subset] = (method, float(tfidf_w), float(semantic_w))
            index.language_strategy = trial
            preds = index.predict_dataframe(
                subset_val, question_col=QUESTION_COL, group_col=LANG_COL, id_col='ID',
                question_embeddings=subset_emb, desc=f'Tune {subset}', log_every=0,
            )
            r1 = compute_rouge(preds, subset_val[ANSWER_COL].tolist())['rouge1_f1']
            if r1 > best_r1:
                best_r1, best_w = r1, tfidf_w
        semantic_w = 1.0 - best_w
        method = 'semantic' if best_w == 0.0 else ('tfidf' if semantic_w == 0.0 else 'hybrid')
        tuned[subset] = (method, float(best_w), float(semantic_w))
        print(f'  {subset}: best tfidf_w={best_w:.2f} (subset R1={best_r1:.4f})')
    index.language_strategy = tuned
    return tuned


def make_submission(ids, predictions, output_path):
    """Build + validate a Zindi submission file."""
    clean_preds = [str(p).strip() for p in predictions]
    sub = pd.DataFrame({
        'ID': ids, 'TargetRLF1': clean_preds, 'TargetR1F1': clean_preds, 'TargetLLM': clean_preds,
    })
    assert list(sub.columns) == ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']
    assert len(sub) == len(test), f"Row mismatch: {len(sub)} vs {len(test)}"
    assert sub[['TargetRLF1', 'TargetR1F1', 'TargetLLM']].notna().all().all()
    assert set(sub['ID']) == set(sample_df['ID']), "ID set doesn't match SampleSubmission"
    sub.to_csv(output_path, index=False, encoding='utf-8')
    print(f"✅ Submission saved to: {output_path}  | shape={sub.shape}")
    return sub


def export_semantic_submission(experiment_id, output_path, index_kwargs=None, language_strategy=None,
                                 corpus=None, test_embeddings=None):
    """Fit train+val index, predict on Test, write Zindi submission CSV."""
    index_kwargs = dict(index_kwargs or {})
    if language_strategy is not None:
        index_kwargs['language_strategy'] = language_strategy
    corpus = corpus if corpus is not None else pd.concat([train, val], ignore_index=True)
    model_name = index_kwargs.get('model_name', SEMANTIC_MODEL_NAME)
    query_prefix = index_kwargs.get('query_encode_prefix', '')

    print(f'\n=== {experiment_id}: test export -> {output_path} ===')
    index = SemanticRoutingIndex(**index_kwargs).fit(corpus.copy())

    if test_embeddings is None:
        test_qs = test[TEST_QUESTION_COL].fillna('').astype(str).tolist()
        print(f'  Encoding {len(test_qs):,} test questions ({model_name})...')
        test_embeddings = encode_questions_for_model(model_name, test_qs, query_prefix=query_prefix)

    preds = index.predict_dataframe(
        test, question_col=TEST_QUESTION_COL, group_col=TEST_LANG_COL, id_col=None,
        question_embeddings=test_embeddings, desc=f'Test {experiment_id}', log_every=0,
    )
    make_submission(test[TEST_ID_COL].values, preds, output_path)
    print(f'  Median ans len: {np.median([len(str(a)) for a in preds]):.0f} chars')
    return preds, index


print("Experiment runner helpers ready: eval_semantic_experiment, tune_hybrid_weights, export_semantic_submission, make_submission")

Experiment runner helpers ready: eval_semantic_experiment, tune_hybrid_weights, export_semantic_submission, make_submission


## Section 8 — Compute Val Embeddings Once Per Session

Run this once at the start of each session. If a checkpoint was restored, it loads instantly. Otherwise it encodes ~6,686 val questions (a few minutes on GPU, longer on CPU) and caches the result for every experiment below.

In [12]:
val_embeddings = load_or_compute_val_embeddings()
print(f"val_embeddings shape: {val_embeddings.shape}")

Encoding 6,686 val questions with sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ...
  Loading encoder (shared, cached): sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 (cuda)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Done in 0.2 min.
Cached to /kaggle/working/checkpoints/embeddings_cache.pkl for future sessions.
val_embeddings shape: (6686, 384)


---
## Experiment E01 — TF-IDF Only Baseline (~10 min)

**Change:** Pure lexical (TF-IDF) retrieval, no semantic embeddings at all.
**Rationale:** Establish the cheapest possible floor before adding any embedding cost — tells us how much semantic similarity is actually buying us later.

In [13]:
e01_strategy = {subset: ('tfidf', 1.0, 0.0) for subset in train[LANG_COL].unique()}

metrics_e01, index_e01 = eval_semantic_experiment(
    experiment_id='E01',
    name='TF-IDF only baseline',
    change='tfidf_w=1.0, semantic_w=0.0 for all subsets (no embeddings used)',
    rationale='Cheapest possible floor — measures pure lexical overlap retrieval before paying for embeddings.',
    language_strategy=e01_strategy,
    val_embeddings=val_embeddings,  # unused when tfidf_w=1.0, harmless to pass
    category='retrieval',
)

EXPERIMENT_CONFIGS['E01'] = {'language_strategy': e01_strategy, 'index_kwargs': {}}

  Encoding 29,814 questions...
    Encoded 29,814 / 29,814
  Built semantic index: 29,814 rows, 8 subsets


Val E01:   0%|          | 0/6686 [00:00<?, ?it/s]


=== E01: TF-IDF only baseline ===
ROUGE-1 F1: 0.3927
ROUGE-L F1: 0.3360
 subset    n  rouge1_f1  rougeL_f1
Eng_Uga 1688   0.481750   0.429424
Aka_Gha 1114   0.279621   0.167338
Eng_Gha 1104   0.255666   0.172279
Lug_Uga  846   0.424596   0.399512
Eng_Eth  564   0.537299   0.520185
Swa_Ken  518   0.532054   0.491384
Amh_Eth  462   0.145412   0.136491
Eng_Ken  390   0.547243   0.501844
Predictions shorter than 60 chars: 126/6686
Logged E01: TF-IDF only baseline | ROUGE-1=0.3927 ROUGE-L=0.336


---
## Experiment E02 — Semantic Only (~15 min)

**Change:** Pure sentence-embedding cosine similarity, no TF-IDF.
**Rationale:** Isolate how much the multilingual sentence encoder alone helps vs E01's lexical-only floor.

In [14]:
e02_strategy = {subset: ('semantic', 0.0, 1.0) for subset in train[LANG_COL].unique()}

metrics_e02, index_e02 = eval_semantic_experiment(
    experiment_id='E02',
    name='Semantic-only retrieval',
    change='tfidf_w=0.0, semantic_w=1.0 for all subsets (paraphrase-multilingual-MiniLM embeddings)',
    rationale='Isolates the multilingual sentence encoder signal in isolation from lexical overlap.',
    language_strategy=e02_strategy,
    val_embeddings=val_embeddings,
    category='retrieval',
)

EXPERIMENT_CONFIGS['E02'] = {'language_strategy': e02_strategy, 'index_kwargs': {}}

  Encoding 29,814 questions...
    Encoded 29,814 / 29,814
  Built semantic index: 29,814 rows, 8 subsets


Val E02:   0%|          | 0/6686 [00:00<?, ?it/s]


=== E02: Semantic-only retrieval ===
ROUGE-1 F1: 0.4134
ROUGE-L F1: 0.3608
 subset    n  rouge1_f1  rougeL_f1
Eng_Uga 1688   0.658407   0.621963
Aka_Gha 1114   0.258065   0.156092
Eng_Gha 1104   0.269389   0.176696
Lug_Uga  846   0.239227   0.209181
Eng_Eth  564   0.549816   0.529987
Swa_Ken  518   0.406900   0.358074
Amh_Eth  462   0.116399   0.107910
Eng_Ken  390   0.745337   0.724249
Predictions shorter than 60 chars: 117/6686
Logged E02: Semantic-only retrieval | ROUGE-1=0.4134 ROUGE-L=0.3608


---
## Experiment E03 — Fixed-Weight Hybrid (~15 min)

**Change:** Blend TF-IDF (0.35) + semantic (0.65) for every subset — the default `LANGUAGE_STRATEGY`.
**Rationale:** Test whether combining both signals beats either alone (E01, E02).

In [15]:
metrics_e03, index_e03 = eval_semantic_experiment(
    experiment_id='E03',
    name='Fixed-weight hybrid (0.35 tfidf / 0.65 semantic)',
    change='Blend both signals with fixed weights across all subsets',
    rationale='Tests whether combining lexical + semantic beats either single signal from E01/E02.',
    language_strategy=LANGUAGE_STRATEGY,
    val_embeddings=val_embeddings,
    category='retrieval',
)

EXPERIMENT_CONFIGS['E03'] = {'language_strategy': dict(LANGUAGE_STRATEGY), 'index_kwargs': {}}

  Encoding 29,814 questions...
    Encoded 29,814 / 29,814
  Built semantic index: 29,814 rows, 8 subsets


Val E03:   0%|          | 0/6686 [00:00<?, ?it/s]


=== E03: Fixed-weight hybrid (0.35 tfidf / 0.65 semantic) ===
ROUGE-1 F1: 0.4328
ROUGE-L F1: 0.3766
 subset    n  rouge1_f1  rougeL_f1
Eng_Uga 1688   0.572870   0.527290
Aka_Gha 1114   0.281560   0.169344
Eng_Gha 1104   0.281764   0.188050
Lug_Uga  846   0.415287   0.388013
Eng_Eth  564   0.558736   0.539796
Swa_Ken  518   0.551084   0.509083
Amh_Eth  462   0.149637   0.139575
Eng_Ken  390   0.720849   0.693440
Predictions shorter than 60 chars: 117/6686
Logged E03: Fixed-weight hybrid (0.35 tfidf / 0.65 semantic) | ROUGE-1=0.4328 ROUGE-L=0.3766


---
## Experiment E04 — Per-Subset Weight Tuning (~30 min)

**Change:** Grid-search the TF-IDF/semantic blend weight independently for each subset/language, instead of using one fixed weight for all.
**Rationale:** Different languages may favour lexical vs semantic matching differently — e.g. low-resource scripts might do better with TF-IDF if the sentence encoder wasn't trained on them well.

**This is the longest retrieval experiment — fine to split across two sessions if needed** (it iterates subset-by-subset, so you can tune a subset, note the result, and resume later).

In [16]:
subsets_to_tune = list(train[LANG_COL].unique())
print(f"Tuning weights for: {subsets_to_tune}")

base_index = SemanticRoutingIndex(language_strategy=dict(LANGUAGE_STRATEGY)).fit(train.copy())
tuned_strategy = tune_hybrid_weights(base_index, val_embeddings, subsets_to_tune)
print("\nTuned strategy:", tuned_strategy)

Tuning weights for: ['Aka_Gha', 'Amh_Eth', 'Eng_Eth', 'Eng_Gha', 'Eng_Ken', 'Eng_Uga', 'Lug_Uga', 'Swa_Ken']
  Encoding 29,814 questions...
    Encoded 29,814 / 29,814
  Built semantic index: 29,814 rows, 8 subsets


Tune Aka_Gha:   0%|          | 0/1114 [00:00<?, ?it/s]

Tune Aka_Gha:   0%|          | 0/1114 [00:00<?, ?it/s]

Tune Aka_Gha:   0%|          | 0/1114 [00:00<?, ?it/s]

Tune Aka_Gha:   0%|          | 0/1114 [00:00<?, ?it/s]

...

Tune Swa_Ken:   0%|          | 0/518 [00:00<?, ?it/s]

Tune Swa_Ken:   0%|          | 0/518 [00:00<?, ?it/s]

Tune Swa_Ken:   0%|          | 0/518 [00:00<?, ?it/s]

Tune Swa_Ken:   0%|          | 0/518 [00:00<?, ?it/s]

  Swa_Ken: best tfidf_w=0.75 (subset R1=0.5660)

Tuned strategy: {'Aka_Gha': ('hybrid', 0.25, 0.75), 'Amh_Eth': ('hybrid', 0.5, 0.5), 'Eng_Eth': ('hybrid', 0.5, 0.5), 'Eng_Gha': ('hybrid', 0.25, 0.75), 'Eng_Ken': ('semantic', 0.0, 1.0), 'Eng_Uga': ('semantic', 0.0, 1.0), 'Lug_Uga': ('hybrid', 0.75, 0.25), 'Swa_Ken': ('hybrid', 0.75, 0.25)}


In [17]:
metrics_e04, index_e04 = eval_semantic_experiment(
    experiment_id='E04',
    name='Per-subset tuned hybrid weights',
    change='Grid-searched tfidf/semantic weight independently per subset (see tuned_strategy above)',
    rationale='Different languages may favour lexical vs semantic matching differently; per-subset tuning lets each pick its best blend.',
    language_strategy=tuned_strategy,
    val_embeddings=val_embeddings,
    category='retrieval',
    notes=str(tuned_strategy),
)

EXPERIMENT_CONFIGS['E04'] = {'language_strategy': dict(tuned_strategy), 'index_kwargs': {}}

  Encoding 29,814 questions...
    Encoded 29,814 / 29,814
  Built semantic index: 29,814 rows, 8 subsets


Val E04:   0%|          | 0/6686 [00:00<?, ?it/s]


=== E04: Per-subset tuned hybrid weights ===
ROUGE-1 F1: 0.4600
ROUGE-L F1: 0.4071
 subset    n  rouge1_f1  rougeL_f1
Eng_Uga 1688   0.658407   0.621963
Aka_Gha 1114   0.280877   0.169102
Eng_Gha 1104   0.279542   0.186461
Lug_Uga  846   0.437426   0.412697
Eng_Eth  564   0.564066   0.545814
Swa_Ken  518   0.566035   0.526921
Amh_Eth  462   0.152381   0.141657
Eng_Ken  390   0.745337   0.724249
Predictions shorter than 60 chars: 121/6686
Logged E04: Per-subset tuned hybrid weights | ROUGE-1=0.46 ROUGE-L=0.4071


---
## Experiment E05 — Exact-Match Short-Circuit (~10 min)

**Change:** Before doing any retrieval, check if the exact (normalised) question text already exists in the training set — if so, return its answer directly, skipping similarity search.
**Rationale:** If Val/Test contains near-duplicate questions from Train (common in these health-FAQ style datasets), this is free accuracy with near-zero cost.

In [18]:
metrics_e05, index_e05 = eval_semantic_experiment(
    experiment_id='E05',
    name='Exact-match lookup + tuned hybrid fallback',
    change='exact_match_lookup=True, falls back to tuned_strategy from E04 when no exact match exists',
    rationale='Captures free accuracy if Val/Test reuses Train questions verbatim, before paying for similarity search.',
    index_kwargs={'exact_match_lookup': True},
    language_strategy=tuned_strategy if 'tuned_strategy' in dir() else LANGUAGE_STRATEGY,
    val_embeddings=val_embeddings,
    category='retrieval',
)

EXPERIMENT_CONFIGS['E05'] = {
    'language_strategy': tuned_strategy if 'tuned_strategy' in dir() else dict(LANGUAGE_STRATEGY),
    'index_kwargs': {'exact_match_lookup': True},
}

  Encoding 29,814 questions...
    Encoded 29,814 / 29,814
  Built semantic index: 29,814 rows, 8 subsets


Val E05:   0%|          | 0/6686 [00:00<?, ?it/s]


=== E05: Exact-match lookup + tuned hybrid fallback ===
ROUGE-1 F1: 0.4595
ROUGE-L F1: 0.4067
 subset    n  rouge1_f1  rougeL_f1
Eng_Uga 1688   0.657355   0.621005
Aka_Gha 1114   0.280877   0.169102
Eng_Gha 1104   0.279542   0.186461
Lug_Uga  846   0.437426   0.412697
Eng_Eth  564   0.561930   0.543485
Swa_Ken  518   0.566035   0.526921
Amh_Eth  462   0.152381   0.141657
Eng_Ken  390   0.745337   0.724249
Predictions shorter than 60 chars: 122/6686
Logged E05: Exact-match lookup + tuned hybrid fallback | ROUGE-1=0.4595 ROUGE-L=0.4067


---
## Experiment E06 — Deduplicate Near-Identical Training Questions (~10 min)

**Change:** Drop duplicate (subset, normalised-question) rows from the training corpus before indexing.
**Rationale:** Repeated identical questions in Train can bias the index toward over-represented phrasings; dedup tests whether a cleaner, smaller index actually retrieves better matches.

In [19]:
metrics_e06, index_e06 = eval_semantic_experiment(
    experiment_id='E06',
    name='Deduplicated training corpus',
    change='dedup_questions=True — drops duplicate (subset, question) rows before fitting',
    rationale='Tests whether removing redundant near-identical questions improves retrieval quality, not just speed.',
    index_kwargs={'dedup_questions': True},
    language_strategy=tuned_strategy if 'tuned_strategy' in dir() else LANGUAGE_STRATEGY,
    val_embeddings=val_embeddings,
    category='preprocessing',
)

EXPERIMENT_CONFIGS['E06'] = {
    'language_strategy': tuned_strategy if 'tuned_strategy' in dir() else dict(LANGUAGE_STRATEGY),
    'index_kwargs': {'dedup_questions': True},
}

  Encoding 28,832 questions...
    Encoded 28,832 / 28,832
  Built semantic index: 28,832 rows, 8 subsets


Val E06:   0%|          | 0/6686 [00:00<?, ?it/s]


=== E06: Deduplicated training corpus ===
ROUGE-1 F1: 0.4599
ROUGE-L F1: 0.4071
 subset    n  rouge1_f1  rougeL_f1
Eng_Uga 1688   0.658407   0.621963
Aka_Gha 1114   0.280877   0.169102
Eng_Gha 1104   0.279542   0.186461
Lug_Uga  846   0.437426   0.412697
Eng_Eth  564   0.563458   0.545168
Swa_Ken  518   0.566035   0.526921
Amh_Eth  462   0.152381   0.141657
Eng_Ken  390   0.745337   0.724249
Predictions shorter than 60 chars: 120/6686
Logged E06: Deduplicated training corpus | ROUGE-1=0.4599 ROUGE-L=0.4071


---
## Experiment E07 — Character-Level TF-IDF for Low-Resource Subsets (~15 min)

**Change:** Use character n-grams (3-5 chars) instead of word n-grams for TF-IDF on the smallest, most morphologically rich subsets (Amharic, Akan).
**Rationale:** Word-level TF-IDF struggles with agglutinative/morphologically rich languages where word forms vary a lot; character n-grams are more robust to small spelling/inflection differences.

In [20]:
# Identify your smallest / most morphologically complex subsets from the EDA —
# Amharic (Amh_Eth) and Akan (Aka_Gha) are good first candidates.
char_tfidf_targets = [s for s in train[LANG_COL].unique() if s.startswith(('Amh', 'Aka'))]
print(f"Using char-level TF-IDF for: {char_tfidf_targets}")

metrics_e07, index_e07 = eval_semantic_experiment(
    experiment_id='E07',
    name='Char-level TF-IDF for low-resource subsets',
    change=f'char_tfidf_subsets={char_tfidf_targets} (3-5 char n-grams instead of word n-grams)',
    rationale='Character n-grams are more robust to morphological variation in agglutinative low-resource languages than word-level TF-IDF.',
    index_kwargs={'char_tfidf_subsets': char_tfidf_targets},
    language_strategy=tuned_strategy if 'tuned_strategy' in dir() else LANGUAGE_STRATEGY,
    val_embeddings=val_embeddings,
    category='preprocessing',
)

EXPERIMENT_CONFIGS['E07'] = {
    'language_strategy': tuned_strategy if 'tuned_strategy' in dir() else dict(LANGUAGE_STRATEGY),
    'index_kwargs': {'char_tfidf_subsets': char_tfidf_targets},
}

Using char-level TF-IDF for: ['Aka_Gha', 'Amh_Eth']
  Encoding 29,814 questions...
    Encoded 29,814 / 29,814
  Built semantic index: 29,814 rows, 8 subsets


Val E07:   0%|          | 0/6686 [00:00<?, ?it/s]


=== E07: Char-level TF-IDF for low-resource subsets ===
ROUGE-1 F1: 0.4610
ROUGE-L F1: 0.4078
 subset    n  rouge1_f1  rougeL_f1
Eng_Uga 1688   0.658407   0.621963
Aka_Gha 1114   0.283839   0.170340
Eng_Gha 1104   0.279542   0.186461
Lug_Uga  846   0.437426   0.412697
Eng_Eth  564   0.564066   0.545814
Swa_Ken  518   0.566035   0.526921
Amh_Eth  462   0.159972   0.148913
Eng_Ken  390   0.745337   0.724249
Predictions shorter than 60 chars: 115/6686
Logged E07: Char-level TF-IDF for low-resource subsets | ROUGE-1=0.461 ROUGE-L=0.4078


---
## Experiment E08 — Cross-Subset Fallback (~15 min)

**Change:** For subsets with very few training examples, allow retrieval to additionally search a related, larger subset (e.g. a native-language subset falling back to its paired English subset) when the same-subset match is weak.
**Rationale:** Tiny subsets (Amh_Eth has the fewest rows) have a small candidate pool — borrowing candidates from a linked subset increases coverage at some risk of cross-lingual mismatch.

In [21]:
# Map each low-resource subset to a larger related subset sharing the same country grouping.
# Adjust this mapping after checking train[LANG_COL].value_counts() from your EDA.
cross_fallback_map = {
    'Amh_Eth': 'Eng_Eth',
    'Aka_Gha': 'Eng_Gha',
    'Swa_Ken': 'Eng_Ken',
    'Lug_Uga': 'Eng_Uga',
}
print(f"Cross-subset fallback map: {cross_fallback_map}")

metrics_e08, index_e08 = eval_semantic_experiment(
    experiment_id='E08',
    name='Cross-subset fallback for low-resource languages',
    change=f'cross_subset_fallback={cross_fallback_map} — widens candidate pool for small subsets',
    rationale='Small subsets have few candidates to retrieve from; borrowing from a paired larger subset increases match coverage.',
    index_kwargs={'cross_subset_fallback': cross_fallback_map},
    language_strategy=tuned_strategy if 'tuned_strategy' in dir() else LANGUAGE_STRATEGY,
    val_embeddings=val_embeddings,
    category='retrieval',
)

EXPERIMENT_CONFIGS['E08'] = {
    'language_strategy': tuned_strategy if 'tuned_strategy' in dir() else dict(LANGUAGE_STRATEGY),
    'index_kwargs': {'cross_subset_fallback': cross_fallback_map},
}

Cross-subset fallback map: {'Amh_Eth': 'Eng_Eth', 'Aka_Gha': 'Eng_Gha', 'Swa_Ken': 'Eng_Ken', 'Lug_Uga': 'Eng_Uga'}
  Encoding 29,814 questions...
    Encoded 29,814 / 29,814
  Built semantic index: 29,814 rows, 8 subsets


Val E08:   0%|          | 0/6686 [00:00<?, ?it/s]


=== E08: Cross-subset fallback for low-resource languages ===
ROUGE-1 F1: 0.4590
ROUGE-L F1: 0.4063
 subset    n  rouge1_f1  rougeL_f1
Eng_Uga 1688   0.658407   0.621963
Aka_Gha 1114   0.280846   0.169070
Eng_Gha 1104   0.279542   0.186461
Lug_Uga  846   0.432045   0.407661
Eng_Eth  564   0.564066   0.545814
Swa_Ken  518   0.563804   0.525618
Amh_Eth  462   0.150946   0.140710
Eng_Ken  390   0.745337   0.724249
Predictions shorter than 60 chars: 116/6686
Logged E08: Cross-subset fallback for low-resource languages | ROUGE-1=0.459 ROUGE-L=0.4063


---
## Experiment E09 — Short mT5 Fine-Tune (~40 min, needs GPU)

**Change:** Fine-tune `google/mt5-small` for 2 epochs on a 25% sample of Train, evaluated on Val.
**Rationale:** The rubric explicitly wants evidence you understand the *fine-tuning* workflow, not just retrieval. This stays small and fast on purpose — it's a demonstration experiment, not your main strategy.

**Run this only when you have GPU quota available.** It's independent of all the retrieval cells above — skip it entirely on sessions without GPU access, and your retrieval experiments are unaffected.

### Free GPU memory before fine-tuning

E01–E08 cached sentence-transformer encoder(s) on the GPU for reuse across retrieval experiments. Before loading `mt5-small` for E09, explicitly release them — otherwise the encoder(s) can occupy several GB that mt5 training then has no room to use.

In [22]:
release_shared_encoders()

if torch.cuda.is_available():
    free_mb = torch.cuda.mem_get_info()[0] / 1e6
    total_mb = torch.cuda.mem_get_info()[1] / 1e6
    print(f"GPU memory free: {free_mb:,.0f} MB / {total_mb:,.0f} MB total")

Released all cached sentence-transformer encoders from GPU memory.
GPU memory free: 14,991 MB / 15,636 MB total


In [23]:
# Only run this cell if GPU is available this session
if not torch.cuda.is_available():
    print("⚠️  No GPU available this session — skip E09 for now, come back when GPU quota refreshes.")
else:
    print(f"GPU available: {torch.cuda.get_device_name(0)} — proceeding with E09.")

GPU available: Tesla T4 — proceeding with E09.


In [24]:
%%capture
!pip install -q transformers datasets accelerate

In [25]:
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer,
    Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback,
)
from datasets import Dataset as HFDataset

# GPU restriction happens ONCE, at the very top of this notebook (Section 1) via
# os.environ["CUDA_VISIBLE_DEVICES"] = "0" set BEFORE `import torch` was first run.
# That's the only point where it takes effect. As long as you did a full kernel
# restart and ran every cell from the top in order, `device` (defined in Section 1)
# already correctly points at the single visible GPU — no extra device object needed here.

MT5_MODEL_NAME = "google/mt5-small"
MT5_MAX_INPUT  = 256
MT5_MAX_TARGET = 256
MT5_BATCH_SIZE = 4
MT5_GRAD_ACCUM = 4
MT5_EPOCHS     = 1
MT5_SAMPLE_FRAC = 0.25
MT5_EVAL_SAMPLE_SIZE = 300  # subsample of Val.csv used for in-training eval only
MT5_OUTPUT_DIR = str(CKPT_DIR / 'mt5_e09')

mt5_train_df = train.sample(frac=MT5_SAMPLE_FRAC, random_state=SEED).reset_index(drop=True)
print(f"E09 training rows: {len(mt5_train_df):,} ({MT5_SAMPLE_FRAC:.0%} of Train)")

def build_prompt(question, subset_code=None):
    return f"answer health question: {question}"

mt5_train_df = mt5_train_df.copy()
mt5_train_df['input_text'] = mt5_train_df[QUESTION_COL].map(build_prompt)
mt5_train_df['target_text'] = mt5_train_df[ANSWER_COL]

mt5_val_df = val.sample(n=min(MT5_EVAL_SAMPLE_SIZE, len(val)), random_state=SEED).reset_index(drop=True)
mt5_val_df['input_text'] = mt5_val_df[QUESTION_COL].map(build_prompt)
mt5_val_df['target_text'] = mt5_val_df[ANSWER_COL]
print(f"E09 eval rows: {len(mt5_val_df):,} (subsampled from {len(val):,} for speed)")

E09 training rows: 7,454 (25% of Train)
E09 eval rows: 300 (subsampled from 6,686 for speed)


In [26]:
mt5_tokenizer = AutoTokenizer.from_pretrained(MT5_MODEL_NAME)

def mt5_preprocess(examples):
    model_inputs = mt5_tokenizer(examples['input_text'], max_length=MT5_MAX_INPUT, truncation=True, padding=False)
    labels = mt5_tokenizer(text_target=examples['target_text'], max_length=MT5_MAX_TARGET, truncation=True, padding=False)
    labels['input_ids'] = [[(l if l != mt5_tokenizer.pad_token_id else -100) for l in label] for label in labels['input_ids']]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

def df_to_hf(df):
    cols = ['input_text', 'target_text']
    hf = HFDataset.from_pandas(df[cols].reset_index(drop=True))
    return hf.map(mt5_preprocess, batched=True, remove_columns=cols)

mt5_train_ds = df_to_hf(mt5_train_df)
mt5_val_ds   = df_to_hf(mt5_val_df)
print("Tokenised.")

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Map:   0%|          | 0/7454 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenised.


In [27]:
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"torch sees {torch.cuda.device_count()} GPU(s) this session (should be 1).")

mt5_model = AutoModelForSeq2SeqLM.from_pretrained(MT5_MODEL_NAME, tie_word_embeddings=True)
mt5_model = mt5_model.to(device)

# Cheap safety net: if something still wrapped this in DataParallel, unwrap it.
if isinstance(mt5_model, torch.nn.DataParallel):
    mt5_model = mt5_model.module.to(device)

print(f"Loaded {MT5_MODEL_NAME} on {device}: {sum(p.numel() for p in mt5_model.parameters()):,} params")

torch sees 1 GPU(s) this session (should be 1).


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded google/mt5-small on cuda: 556,291,456 params


In [28]:
# GPU visibility is controlled at the very top of this notebook (Section 1,
# `os.environ["CUDA_VISIBLE_DEVICES"] = "0"` set BEFORE `import torch`). That is the
# only point where this setting actually takes effect — CUDA reads it once at
# initialisation and ignores changes afterward. This means: if you restarted the
# kernel and ran every cell from the top in order, only GPU 0 is visible here.
#
# We deliberately do NOT monkey-patch torch.cuda.device_count() (an earlier attempt
# at this caused infinite recursion, since is_available() calls device_count()
# internally) — the env var is the correct and only reliable mechanism.

visible_gpus = torch.cuda.device_count()
print(f"torch sees {visible_gpus} GPU(s) this session.")

if visible_gpus > 1:
    raise RuntimeError(
        "More than one GPU is visible to this process, which means "
        "CUDA_VISIBLE_DEVICES wasn't set before CUDA initialised. "
        "Fix: Kernel -> Restart (a full restart, not just re-running cells), "
        "then run every cell from the very top of the notebook in order — "
        "do not skip ahead to this cell after a partial run."
    )

def mt5_compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.asarray(predictions)
    labels = np.asarray(labels)
    vocab_size = len(mt5_tokenizer)
    predictions = np.where((predictions < 0) | (predictions >= vocab_size), mt5_tokenizer.pad_token_id, predictions)
    labels = np.where(labels != -100, labels, mt5_tokenizer.pad_token_id)
    labels = np.where((labels < 0) | (labels >= vocab_size), mt5_tokenizer.pad_token_id, labels)
    decoded_preds = mt5_tokenizer.batch_decode(predictions.tolist(), skip_special_tokens=True)
    decoded_labels = mt5_tokenizer.batch_decode(labels.tolist(), skip_special_tokens=True)
    return compute_rouge(decoded_preds, decoded_labels)

mt5_collator = DataCollatorForSeq2Seq(tokenizer=mt5_tokenizer, model=mt5_model, padding=True, pad_to_multiple_of=8)

mt5_args = Seq2SeqTrainingArguments(
    output_dir=MT5_OUTPUT_DIR,
    num_train_epochs=MT5_EPOCHS,
    per_device_train_batch_size=MT5_BATCH_SIZE,
    per_device_eval_batch_size=max(1, MT5_BATCH_SIZE // 2),
    gradient_accumulation_steps=MT5_GRAD_ACCUM,
    learning_rate=3e-4,
    warmup_steps=50,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='rouge1_f1',
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=MT5_MAX_TARGET,
    generation_num_beams=1,
    # fp16 disabled: T5/mT5's layer-norm structure is well-documented to produce
    # NaN losses under fp16 mixed precision (this is what caused E09's eval_loss=nan,
    # eval_rouge1_f1=0.0 — the model wasn't broken, the precision setting was unstable).
    # mt5-small is tiny enough that full fp32 isn't a memory problem.
    fp16=False,
    gradient_checkpointing=True,
    logging_steps=50,
    save_total_limit=1,
    seed=SEED,
    report_to='none',
)

mt5_trainer = Seq2SeqTrainer(
    model=mt5_model, args=mt5_args, train_dataset=mt5_train_ds, eval_dataset=mt5_val_ds,
    processing_class=mt5_tokenizer, data_collator=mt5_collator, compute_metrics=mt5_compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

import threading

# HARD TIMEOUT GUARD: a previous overnight run silently hung for 5+ hours during
# training with no error and no progress (a CUDA call appears to have stalled rather
# than crashed — this can happen under infra contention or memory fragmentation).
# Normal runtime for this exact config is ~24 minutes. We cap at 60 minutes — generous
# headroom over normal, but small enough that an overnight unattended run fails loudly
# and quickly instead of burning your whole GPU quota on a stall.
MT5_TRAIN_TIMEOUT_SEC = 60 * 60  # 60 minutes

_train_result_holder = {}
_train_exception_holder = {}

def _run_training():
    try:
        _train_result_holder['result'] = mt5_trainer.train()
    except Exception as e:
        _train_exception_holder['error'] = e

t0 = time.time()
_train_thread = threading.Thread(target=_run_training, daemon=True)
_train_thread.start()
_train_thread.join(timeout=MT5_TRAIN_TIMEOUT_SEC)

if _train_thread.is_alive():
    raise TimeoutError(
        f"E09 training exceeded {MT5_TRAIN_TIMEOUT_SEC / 60:.0f} minutes "
        f"(normal runtime is ~24 min) and appears hung, not just slow. "
        f"This matches a known failure mode where a CUDA call stalls silently "
        f"with no error and no progress. Recommended fix: cancel this session "
        f"entirely, start a genuinely fresh session (not just kernel restart), "
        f"and re-run. If it hangs again on a fresh session, it may be Kaggle "
        f"infrastructure congestion rather than a notebook bug — try again later "
        f"or reduce MT5_SAMPLE_FRAC / MT5_EPOCHS further."
    )

if 'error' in _train_exception_holder:
    raise _train_exception_holder['error']

mt5_result = _train_result_holder['result']
mt5_runtime_min = (time.time() - t0) / 60
print(f"Training took {mt5_runtime_min:.1f} min")
print(mt5_result.metrics)

torch sees 1 GPU(s) this session.


Epoch,Training Loss,Validation Loss,Rouge1 F1,Rougel F1
1,14.800613,3.021876,0.079747,0.071097


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training took 19.1 min
{'train_runtime': 1143.7915, 'train_samples_per_second': 6.517, 'train_steps_per_second': 0.407, 'total_flos': 462969924648960.0, 'train_loss': 22.07996044240796, 'epoch': 1.0}


In [29]:
# Same hard-timeout pattern as training — this is actually where your last overnight
# run hung silently (training finished and printed its summary, but evaluate() never
# returned and never errored). Cap generously above normal (~5 min) at 30 minutes.
MT5_EVAL_TIMEOUT_SEC = 30 * 60

_eval_result_holder = {}
_eval_exception_holder = {}

def _run_eval():
    try:
        _eval_result_holder['result'] = mt5_trainer.evaluate()
    except Exception as e:
        _eval_exception_holder['error'] = e

_eval_thread = threading.Thread(target=_run_eval, daemon=True)
_eval_thread.start()
_eval_thread.join(timeout=MT5_EVAL_TIMEOUT_SEC)

if _eval_thread.is_alive():
    raise TimeoutError(
        f"E09 evaluation exceeded {MT5_EVAL_TIMEOUT_SEC / 60:.0f} minutes and appears "
        f"hung (normal runtime is ~5 min). This is the exact failure mode seen in an "
        f"earlier overnight run: training completed and printed its summary, but "
        f"evaluate()'s generation loop over Val.csv never returned. Recommended fix: "
        f"cancel this session, start a genuinely fresh session, and re-run."
    )

if 'error' in _eval_exception_holder:
    raise _eval_exception_holder['error']

mt5_eval_metrics = _eval_result_holder['result']
print(mt5_eval_metrics)

tracker.log(
    experiment_id='E09',
    name='mT5-small fine-tune, 2 epochs, 25% data',
    category='fine-tuning',
    change=f'Fine-tuned {MT5_MODEL_NAME} for {MT5_EPOCHS} epochs on {MT5_SAMPLE_FRAC:.0%} of Train',
    rationale='Demonstrates the fine-tuning workflow alongside the retrieval approach, kept short to fit a single GPU session.',
    rouge1=round(mt5_eval_metrics.get('eval_rouge1_f1', 0), 4),
    rougel=round(mt5_eval_metrics.get('eval_rougeL_f1', 0), 4),
    runtime_min=round(mt5_runtime_min, 1),
    notes=f'epochs={MT5_EPOCHS}, sample_frac={MT5_SAMPLE_FRAC}, batch={MT5_BATCH_SIZE}x{MT5_GRAD_ACCUM}',
)

{'eval_loss': 3.0218758583068848, 'eval_rouge1_f1': 0.07974682761397155, 'eval_rougeL_f1': 0.07109713857990975, 'eval_runtime': 357.1698, 'eval_samples_per_second': 0.84, 'eval_steps_per_second': 0.42, 'epoch': 1.0}
Logged E09: mT5-small fine-tune, 2 epochs, 25% data | ROUGE-1=0.0797 ROUGE-L=0.0711


---
## Experiment E10 — Compare Best Retrieval vs mT5, Pick Winner (~20 min)

**Change:** Compare your best-performing retrieval config (highest ROUGE-1 among E01–E08) against the E09 mT5 fine-tune on the same Val set, side by side per subset.
**Rationale:** Closes the loop — shows you didn't just run experiments in isolation, but used the results to make a final, justified choice for your submission.

In [30]:
tracker.show()

retrieval_rows = tracker.df[tracker.df['category'].isin(['retrieval', 'preprocessing'])]
best_retrieval_row = retrieval_rows.sort_values('rouge1', ascending=False).iloc[0]
print(f"\nBest retrieval experiment so far: {best_retrieval_row['experiment_id']} "
      f"({best_retrieval_row['name']}) — ROUGE-1={best_retrieval_row['rouge1']}")

mt5_rows = tracker.df[tracker.df['category'] == 'fine-tuning']
if len(mt5_rows) > 0:
    best_mt5_row = mt5_rows.sort_values('rouge1', ascending=False).iloc[0]
    print(f"Best fine-tuning experiment: {best_mt5_row['experiment_id']} — ROUGE-1={best_mt5_row['rouge1']}")
    winner = best_retrieval_row if best_retrieval_row['rouge1'] >= best_mt5_row['rouge1'] else best_mt5_row
else:
    print("No mT5 experiment logged yet — comparing retrieval experiments only.")
    winner = best_retrieval_row

print(f"\n🏆 Winner: {winner['experiment_id']} ({winner['name']}) — use this for your final submission.")

experiment_id                                             name  rouge1  rougel lb_score  runtime_min
          E01                             TF-IDF only baseline  0.3927  0.3360     None          2.0
          E02                          Semantic-only retrieval  0.4134  0.3608     None          1.5
          E03 Fixed-weight hybrid (0.35 tfidf / 0.65 semantic)  0.4328  0.3766     None          2.1
          E04                  Per-subset tuned hybrid weights  0.4600  0.4071     None          1.9
          E05       Exact-match lookup + tuned hybrid fallback  0.4595  0.4067     None          1.9
          E06                     Deduplicated training corpus  0.4599  0.4071     None          1.9
          E07       Char-level TF-IDF for low-resource subsets  0.4610  0.4078     None          3.0
          E08 Cross-subset fallback for low-resource languages  0.4590  0.4063     None          2.7
          E09          mT5-small fine-tune, 2 epochs, 25% data  0.0797  0.0711     None    

In [31]:
tracker.log(
    experiment_id='E10',
    name='Final comparison: best retrieval vs mT5 fine-tune',
    category='analysis',
    change='No new model — compares all prior experiments to select the final submission config',
    rationale='Closes the experimentation loop by explicitly justifying the final model choice from measured Val performance rather than assumption.',
    rouge1=winner['rouge1'],
    rougel=winner['rougel'],
    notes=f"Winner: {winner['experiment_id']} ({winner['name']})",
)

Logged E10: Final comparison: best retrieval vs mT5 fine-tune | ROUGE-1=0.461 ROUGE-L=0.4078


---
## Section 9 — Save Checkpoint (run at the end of every session)

This bundles your experiment tracker CSV + cached val embeddings into one zip. **Download this zip and re-upload it as a new version of a Kaggle Dataset** (e.g. name it `health-qa-checkpoint`). Next session, attach that dataset via **Add Input** and Section 3b will find and restore it automatically — no re-encoding, no lost experiment history.

In [32]:
CHECKPOINT_ZIP_PATH = f'{WORKING_DIR}/health_qa_checkpoint.zip'

shutil.make_archive(
    base_name=CHECKPOINT_ZIP_PATH.replace('.zip', ''),
    format='zip',
    root_dir=str(CKPT_DIR.parent),
    base_dir=CKPT_DIR.name,
)

print(f"✅ Checkpoint zipped to: {CHECKPOINT_ZIP_PATH}")
print(f"   Size: {os.path.getsize(CHECKPOINT_ZIP_PATH) / 1e6:.1f} MB")
print("\nContents:")
for f in sorted(os.listdir(CKPT_DIR)):
    size = os.path.getsize(CKPT_DIR / f)
    print(f"   {f}  ({size/1e6:.1f} MB)")

print("\n📋 Next steps:")
print("  1. Download health_qa_checkpoint.zip from the output panel")
print("  2. Go to Kaggle -> Datasets -> New Dataset (or 'New Version' if you already made one)")
print("  3. Upload this zip (Kaggle will auto-extract it, or upload as-is)")
print("  4. Next session: Add Input -> attach that dataset -> re-run Section 3b")

✅ Checkpoint zipped to: /kaggle/working/health_qa_checkpoint.zip
   Size: 2786.4 MB

Contents:
   embeddings_cache.pkl  (10.3 MB)
   experiment_tracker.csv  (0.0 MB)
   mt5_e09  (0.0 MB)

📋 Next steps:
  1. Download health_qa_checkpoint.zip from the output panel
  2. Go to Kaggle -> Datasets -> New Dataset (or 'New Version' if you already made one)
  3. Upload this zip (Kaggle will auto-extract it, or upload as-is)
  4. Next session: Add Input -> attach that dataset -> re-run Section 3b


In [33]:
from IPython.display import FileLink
FileLink(CHECKPOINT_ZIP_PATH)

/kaggle/working/health_qa_checkpoint.zip

### Experiment Config Registry

`EXPERIMENT_CONFIGS` was initialised back in Section 4 and has been filling up automatically as you ran E01–E08 above — each experiment cell registers its own config. If you're resuming a session and a config is missing, just re-run that experiment's cell once.

### Helper: Generate a Submission for Any Experiment by ID

Instead of manually copying `language_strategy`/`index_kwargs` into Section 10 each time, call `submit_experiment('E04')` (or whichever ID) — it looks up that experiment's exact config from the registry above and writes a uniquely-named file: `submission_E04.csv`. Run it once per experiment you want to actually upload to Zindi.

In [34]:
def submit_experiment(experiment_id, corpus=None, test_embeddings=None):
    """Build a uniquely-named submission file for a specific logged experiment.
    Looks up the saved config from EXPERIMENT_CONFIGS — make sure that experiment's
    cell has been run at least once this session so its config is registered."""
    if experiment_id not in EXPERIMENT_CONFIGS:
        raise KeyError(
            f"No saved config for {experiment_id}. Re-run that experiment's cell first "
            f"(this session) so its config gets registered. Available: {list(EXPERIMENT_CONFIGS.keys())}"
        )
    cfg = EXPERIMENT_CONFIGS[experiment_id]
    output_path = f'{WORKING_DIR}/submission_{experiment_id}.csv'
    preds, index = export_semantic_submission(
        experiment_id=experiment_id,
        output_path=output_path,
        index_kwargs=cfg.get('index_kwargs', {}),
        language_strategy=cfg.get('language_strategy'),
        corpus=corpus,
        test_embeddings=test_embeddings,
    )
    print(f"\n📄 Ready to upload: {output_path}")
    return preds, index


print("submit_experiment(experiment_id) ready — e.g. submit_experiment('E04')")

submit_experiment(experiment_id) ready — e.g. submit_experiment('E04')


**Example — generate submissions for a few experiments to show leaderboard progression:**
```python
submit_experiment('E01')   # -> submission_E01.csv  (baseline)
submit_experiment('E04')   # -> submission_E04.csv  (tuned weights)
submit_experiment('E08')   # -> submission_E08.csv  (cross-subset fallback)
```
Upload each `.csv` to Zindi separately, screenshot the score after each one — that's your leaderboard progression evidence for the report.

In [35]:
# Uncomment and edit with the experiment IDs you actually want to submit:

# submit_experiment('E01')
# submit_experiment('E04')
# submit_experiment('E08')

---
## Automatic Submission Export (runs unattended)

This cell loops through **every retrieval experiment that has a registered config** (E01–E08) and exports a separate `submission_<ID>.csv` for each one — no manual `submit_experiment('E04')` calls needed. Designed to run as the last step before you close the laptop: by the time you wake up, every file is sitting in `/kaggle/working/`, ready to upload to Zindi.

**Note on mT5 (E09):** this loop only exports *retrieval* experiments, since mT5 inference needs the fine-tuned model reloaded and is a different code path. If E09 ends up winning per the tracker, it's flagged below so you know to handle it separately.

In [36]:
import threading

# Run every registered retrieval experiment through submit_experiment() automatically.
# Each call is wrapped in a hard timeout (not just try/except) — a silent CUDA hang
# during test-set encoding would otherwise block this loop forever even with
# exception handling, since a stall never raises an exception to catch.
SUBMIT_TIMEOUT_SEC = 15 * 60  # generous vs. normal (~1-3 min per retrieval experiment)

AUTO_SUBMIT_RESULTS = {}

print(f"Experiments with registered configs: {list(EXPERIMENT_CONFIGS.keys())}\n")

for exp_id in sorted(EXPERIMENT_CONFIGS.keys()):
    print(f"\n{'=' * 60}\nExporting submission for {exp_id}...\n{'=' * 60}")

    _result_holder = {}
    _exception_holder = {}

    def _run_submit(eid=exp_id):
        try:
            _result_holder['preds'] = submit_experiment(eid)
        except Exception as e:
            _exception_holder['error'] = e

    _thread = threading.Thread(target=_run_submit, daemon=True)
    _thread.start()
    _thread.join(timeout=SUBMIT_TIMEOUT_SEC)

    if _thread.is_alive():
        msg = f"timed out after {SUBMIT_TIMEOUT_SEC / 60:.0f} min (likely hung, not just slow)"
        print(f"⚠️  {exp_id} {msg}")
        AUTO_SUBMIT_RESULTS[exp_id] = msg
        # Don't join() again / don't wait further — move on to the next experiment.
        # The hung thread is daemon=True so it won't block notebook shutdown.
        continue

    if 'error' in _exception_holder:
        print(f"⚠️  {exp_id} failed: {_exception_holder['error']}")
        AUTO_SUBMIT_RESULTS[exp_id] = f"failed: {_exception_holder['error']}"
        continue

    AUTO_SUBMIT_RESULTS[exp_id] = 'success'

print(f"\n\n{'=' * 60}\nSUMMARY\n{'=' * 60}")
for exp_id, status in AUTO_SUBMIT_RESULTS.items():
    icon = '✅' if status == 'success' else '❌'
    print(f"{icon} {exp_id}: {status}")

# Flag if the tracker's overall winner is E09 (mT5), which this loop can't export
if len(tracker.df) > 0:
    retrieval_df = tracker.df[tracker.df['category'].isin(['retrieval', 'preprocessing'])]
    if len(retrieval_df) > 0:
        best_retrieval = retrieval_df.sort_values('rouge1', ascending=False).iloc[0]
        mt5_rows = tracker.df[tracker.df['category'] == 'fine-tuning']
        if len(mt5_rows) > 0:
            best_mt5 = mt5_rows.sort_values('rouge1', ascending=False).iloc[0]
            if best_mt5['rouge1'] > best_retrieval['rouge1']:
                print(
                    f"\n⚠️  NOTE: E09 (mT5, ROUGE-1={best_mt5['rouge1']}) currently beats "
                    f"every retrieval experiment (best: {best_retrieval['experiment_id']}, "
                    f"ROUGE-1={best_retrieval['rouge1']}). mT5 inference on Test.csv isn't "
                    f"included in this auto-export loop — you'll need to run that separately "
                    f"if you want to submit the mT5 model's predictions."
                )

Experiments with registered configs: ['E01', 'E02', 'E03', 'E04', 'E05', 'E06', 'E07', 'E08']


Exporting submission for E01...

=== E01: test export -> /kaggle/working/submission_E01.csv ===
  Encoding 36,500 questions...
  Loading encoder (shared, cached): sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 (cuda)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Encoded 36,500 / 36,500
  Built semantic index: 36,500 rows, 8 subsets
  Encoding 2,618 test questions (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)...


Test E01:   0%|          | 0/2618 [00:00<?, ?it/s]

...

✅ Submission saved to: /kaggle/working/submission_E06.csv  | shape=(2618, 4)
  Median ans len: 475 chars

📄 Ready to upload: /kaggle/working/submission_E06.csv

Exporting submission for E07...

=== E07: test export -> /kaggle/working/submission_E07.csv ===
  Encoding 36,500 questions...
    Encoded 36,500 / 36,500
  Built semantic index: 36,500 rows, 8 subsets
  Encoding 2,618 test questions (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)...


Test E07:   0%|          | 0/2618 [00:00<?, ?it/s]

✅ Submission saved to: /kaggle/working/submission_E07.csv  | shape=(2618, 4)
  Median ans len: 479 chars

📄 Ready to upload: /kaggle/working/submission_E07.csv

Exporting submission for E08...

=== E08: test export -> /kaggle/working/submission_E08.csv ===
  Encoding 36,500 questions...
    Encoded 36,500 / 36,500
  Built semantic index: 36,500 rows, 8 subsets
  Encoding 2,618 test questions (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)...


Test E08:   0%|          | 0/2618 [00:00<?, ?it/s]

✅ Submission saved to: /kaggle/working/submission_E08.csv  | shape=(2618, 4)
  Median ans len: 473 chars

📄 Ready to upload: /kaggle/working/submission_E08.csv


SUMMARY
✅ E01: success
✅ E02: success
✅ E03: success
✅ E04: success
✅ E05: success
✅ E06: success
✅ E07: success
✅ E08: success


### Export the overall winner as `submission_FINAL.csv`

Picks whichever retrieval experiment scored highest and saves it under a fixed filename, so you always know which file to upload first when you wake up.

In [37]:
retrieval_df = tracker.df[tracker.df['category'].isin(['retrieval', 'preprocessing'])]

if len(retrieval_df) > 0:
    best_id = retrieval_df.sort_values('rouge1', ascending=False).iloc[0]['experiment_id']
    print(f"Best retrieval experiment: {best_id} — exporting as submission_FINAL.csv\n")
    preds, _ = submit_experiment(best_id)
    shutil.copy(f'{WORKING_DIR}/submission_{best_id}.csv', f'{WORKING_DIR}/submission_FINAL.csv')
    print(f"\n✅ submission_FINAL.csv ready (copy of {best_id}'s submission)")
else:
    print("No retrieval experiments found in tracker — nothing to export.")

print(f"\n=== All submission files in {WORKING_DIR} ===")
for f in sorted(os.listdir(WORKING_DIR)):
    path = f'{WORKING_DIR}/{f}'
    if os.path.isfile(path) and f.startswith('submission'):
        print(f"  {f}  ({os.path.getsize(path)/1e3:.0f} KB)")

Best retrieval experiment: E07 — exporting as submission_FINAL.csv


=== E07: test export -> /kaggle/working/submission_E07.csv ===
  Encoding 36,500 questions...
    Encoded 36,500 / 36,500
  Built semantic index: 36,500 rows, 8 subsets
  Encoding 2,618 test questions (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)...


Test E07:   0%|          | 0/2618 [00:00<?, ?it/s]

✅ Submission saved to: /kaggle/working/submission_E07.csv  | shape=(2618, 4)
  Median ans len: 479 chars

📄 Ready to upload: /kaggle/working/submission_E07.csv

✅ submission_FINAL.csv ready (copy of E07's submission)

=== All submission files in /kaggle/working ===
  submission_E01.csv  (4581 KB)
  submission_E02.csv  (4652 KB)
  submission_E03.csv  (4607 KB)
  submission_E04.csv  (4623 KB)
  submission_E05.csv  (4621 KB)
  submission_E06.csv  (4623 KB)
  submission_E07.csv  (4657 KB)
  submission_E08.csv  (4616 KB)
  submission_FINAL.csv  (4657 KB)


---
## Section 10 — Final Submission Export

Run this once you've decided on your best config (from E10's comparison). It fits the winning retrieval strategy on **Train + Val combined** (more data = better retrieval candidates for Test, since Val is no longer needed for scoring at this point) and predicts on Test.

In [38]:
# EDIT THIS to match your actual best strategy from the experiment log (e.g. tuned_strategy from E04,
# or e08's cross_fallback_map config, or LANGUAGE_STRATEGY for the simple fixed-weight hybrid).
FINAL_STRATEGY = tuned_strategy if 'tuned_strategy' in dir() else LANGUAGE_STRATEGY
FINAL_INDEX_KWARGS = {}  # e.g. {'exact_match_lookup': True, 'cross_subset_fallback': cross_fallback_map}

final_preds, final_index = export_semantic_submission(
    experiment_id='FINAL',
    output_path=f'{WORKING_DIR}/submission.csv',
    index_kwargs=FINAL_INDEX_KWARGS,
    language_strategy=FINAL_STRATEGY,
)


=== FINAL: test export -> /kaggle/working/submission.csv ===
  Encoding 36,500 questions...
    Encoded 36,500 / 36,500
  Built semantic index: 36,500 rows, 8 subsets
  Encoding 2,618 test questions (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)...


Test FINAL:   0%|          | 0/2618 [00:00<?, ?it/s]

✅ Submission saved to: /kaggle/working/submission.csv  | shape=(2618, 4)
  Median ans len: 475 chars


In [39]:
from IPython.display import FileLink
FileLink(f'{WORKING_DIR}/submission.csv')

/kaggle/working/submission.csv

In [40]:
print("=== Final Experiment Log ===")
tracker.show()
print(f"\nFull tracker saved at: {TRACKER_PATH}")
print("Download this CSV too — you'll paste it into your report's Results section.")

=== Final Experiment Log ===
experiment_id                                              name  rouge1  rougel lb_score  runtime_min
          E01                              TF-IDF only baseline  0.3927  0.3360     None          2.0
          E02                           Semantic-only retrieval  0.4134  0.3608     None          1.5
          E03  Fixed-weight hybrid (0.35 tfidf / 0.65 semantic)  0.4328  0.3766     None          2.1
          E04                   Per-subset tuned hybrid weights  0.4600  0.4071     None          1.9
          E05        Exact-match lookup + tuned hybrid fallback  0.4595  0.4067     None          1.9
          E06                      Deduplicated training corpus  0.4599  0.4071     None          1.9
          E07        Char-level TF-IDF for low-resource subsets  0.4610  0.4078     None          3.0
          E08  Cross-subset fallback for low-resource languages  0.4590  0.4063     None          2.7
          E09           mT5-small fine-tune, 2 epochs